# CRED-HUUNT v2 Kaggle Training And Evaluation Pipeline

This notebook runs the CRED-HUUNT v2 workflow on Kaggle: clone or copy the project, install dependencies, verify GPU, build the v2 dataset, train LoRA adapters, evaluate them, and package outputs for download.

Kaggle settings to use before running:

- Accelerator: GPU, preferably T4 x2 or P100.
- Internet: On if cloning from GitHub or downloading Hugging Face models.
- Add Input: optional if you upload the repository as a Kaggle Dataset instead of cloning from GitHub.

## 1. Configuration

Everything you choose for a run lives in the next cell:

- **Pipeline stages** — `RUN_TRAINING`, `RUN_EVALUATION`, `RUN_STRATEGY_BENCHMARK` switch whole sections on or off.
- **`MODELS`** — set `"enabled": True` for each model you want to train and benchmark. The default is the family-diverse 2-3B trio (`qwen2.5-coder:3b`, `granite3.3:2b`, `llama3.2:3b`).
- **`STRATEGIES`** — the reasoning strategies tested in Section 7.

If you upload the project as a Kaggle Dataset instead of cloning, set `USE_GITHUB_CLONE = False` and point `KAGGLE_INPUT_PROJECT_DIR` at the folder containing `src/`, `scripts/`, and `data/`.

For a first smoke run use `PER_CLASS_CAP = 1000`, `EPOCHS = 1`, and `BENCHMARK_TEST_MODE = True`. For a full run set `PER_CLASS_CAP = 0`, raise `EPOCHS`, and set `EVAL_LIMIT = 0` / `BENCHMARK_LIMIT = 0`.

In [ ]:
from pathlib import Path

USE_GITHUB_CLONE = True
GITHUB_REPO_URL = "https://github.com/MohammedElGuerrouj/Cred_Hunt_v2.git"
KAGGLE_INPUT_PROJECT_DIR = Path("/kaggle/input/cred-hunt-v2")
PROJECT_DIR = Path("/kaggle/working/Cred_Hunt_v2")

# ---- Pipeline stages: turn whole sections on/off ------------------------
RUN_TRAINING = True             # Section 5 - train LoRA adapters
RUN_EVALUATION = True           # Section 6 - evaluate trained adapters
RUN_STRATEGY_BENCHMARK = True   # Section 7 - benchmark reasoning strategies

# ---- Dataset build settings --------------------------------------------
USE_AXA_GENERATOR = True   # True  = generate the AXA synthetic corpus (10 languages,
                           #         ~24 credential types, REVIEW class) and train on it.
                           # False = use the original data/*.crdownload source files.
AXA_SEED = 42              # deterministic seed for the AXA generator
AUGMENT_FP = 3
PER_CLASS_CAP = 0  # 0 keeps the full dataset; use 1000-5000 for a fast smoke run

# ---- Training settings -------------------------------------------------
EPOCHS = 1
BATCH_SIZE = 2
LEARNING_RATE = 5e-4
LOAD_4BIT = False  # QLoRA 4-bit base model. Leave False on Kaggle T4/P100 (>=16GB VRAM).
                   # Set True for <=8GB GPUs (needs the QLoRA --load-4bit support in
                   # scripts/lora_fine_tune.py).
EVAL_LIMIT = 500   # 0 evaluates the full test split

# ---- MODEL SELECTION ----------------------------------------------------
# Set "enabled": True for every model you want to train AND benchmark.
# Family-diverse 2-3B trio - see docs_v2/MODEL_SELECTION.md.
MODELS = [
    {
        "enabled": True,
        "ollama_name": "qwen2.5-coder:3b",
        "hf_name": "Qwen/Qwen2.5-Coder-3B-Instruct",
        "output_dir": "/kaggle/working/lora-qwen25-coder-3b",
    },
    {
        "enabled": False,
        "ollama_name": "granite3.3:2b",
        "hf_name": "ibm-granite/granite-3.3-2b-instruct",
        "output_dir": "/kaggle/working/lora-granite33-2b",
    },
    {
        "enabled": False,
        "ollama_name": "llama3.2:3b",
        "hf_name": "meta-llama/Llama-3.2-3B-Instruct",
        "output_dir": "/kaggle/working/lora-llama32-3b",
    },
]

# ---- REASONING STRATEGY SELECTION ---------------------------------------
# Strategies benchmarked in Section 7 - see docs_v2/REASONING_STRATEGIES.md.
# Comment out any strategy you want to skip.
STRATEGIES = [
    "direct_json",
    "few_shot",
    "self_consistency",
    "cot_distilled",
    "react_triage",
]
BENCHMARK_TEST_MODE = True  # True  = deterministic fake LLM, no Ollama needed (smoke test).
                            # False = real Ollama benchmark (Ollama must be serving the
                            #         enabled models first).
BENCHMARK_LIMIT = 200       # records per (model, strategy) cell; 0 = full test split

enabled_models = [m["ollama_name"] for m in MODELS if m.get("enabled")]
print("Dataset           :", "AXA synthetic generator" if USE_AXA_GENERATOR else "original source files")
print("Enabled models   :", enabled_models or "(none - enable at least one)")
print("Strategies        :", STRATEGIES)
print("Stages            :", {"train": RUN_TRAINING, "eval": RUN_EVALUATION,
                               "benchmark": RUN_STRATEGY_BENCHMARK})

## 2. Get The Project Into `/kaggle/working`

Kaggle input datasets are read-only. This cell creates a writable working copy.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def run_cmd(cmd, cwd=None, check=True):
    print("$", " ".join(str(part) for part in cmd))
    return subprocess.run(cmd, cwd=cwd, check=check)


if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if USE_GITHUB_CLONE:
    run_cmd(["git", "clone", "--depth", "1", GITHUB_REPO_URL, str(PROJECT_DIR)])
else:
    if not KAGGLE_INPUT_PROJECT_DIR.exists():
        raise FileNotFoundError(f"Kaggle input project dir not found: {KAGGLE_INPUT_PROJECT_DIR}")
    shutil.copytree(KAGGLE_INPUT_PROJECT_DIR, PROJECT_DIR)

os.chdir(PROJECT_DIR)
print("Project directory:", Path.cwd())
print("Top-level files:", sorted(path.name for path in PROJECT_DIR.iterdir()))

## 3. Install Dependencies And Verify GPU

Kaggle usually includes CUDA PyTorch already. This installs the project-specific Hugging Face and evaluation packages without replacing PyTorch.

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# bitsandbytes is only needed when LOAD_4BIT = True (QLoRA); harmless otherwise.
run_cmd([sys.executable, "-m", "pip", "install", "-q", "-U",
         "transformers", "peft", "datasets", "accelerate",
         "bitsandbytes", "scikit-learn", "tqdm", "requests"])

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(f"GPU {index}:", torch.cuda.get_device_name(index))

if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU is not enabled. Open Notebook Settings and choose a GPU accelerator.")

## 4. Build The v2 Dataset

When `USE_AXA_GENERATOR = True`, this first runs `scripts/generate_axa_synthetic.py` — which mines AXA vocabulary from `data/true_positive.crdownload` / `data/false_positive.crdownload` and writes a richer synthetic corpus (10 languages, ~24 credential types, `REVIEW` class) to `data/synthetic/`. The processor then consumes that corpus via `--source-*` flags.

When `USE_AXA_GENERATOR = False`, the processor builds directly from the original `data/*.crdownload` files.

Either way, false positives are augmented, records merged, and train/validation/test JSONL files written to `data/`.

In [ ]:
# The AXA generator mines vocabulary from these source files; the processor
# reads them directly when USE_AXA_GENERATOR is False. Either naming
# (true_positive.crdownload or "true_positive (1).crdownload") is accepted.
for stem in ["true_positive", "false_positive"]:
    matches = sorted(Path("data").glob(f"{stem}*.crdownload"))
    if not matches:
        raise FileNotFoundError(f"Missing source dataset: data/{stem}*.crdownload")
    for match in matches:
        print(match, "size=", match.stat().st_size)

In [ ]:
if USE_AXA_GENERATOR:
    # Generate the AXA synthetic corpus (mines vocabulary from the source files).
    run_cmd([
        sys.executable,
        "scripts/generate_axa_synthetic.py",
        "--out-dir", "data/synthetic",
        "--seed", str(AXA_SEED),
    ], cwd=PROJECT_DIR)

process_cmd = [
    sys.executable,
    "scripts/process_synthetic_training_data.py",
    "--target", "both",
    "--augment-fp", str(AUGMENT_FP),
]
if USE_AXA_GENERATOR:
    # --data-dir defaults to "data", so processed JSONL/CSV land in data/ (where
    # Sections 5-6 expect them); --source-* points the inputs at data/synthetic/.
    process_cmd += [
        "--source-tp", "data/synthetic/true_positive.crdownload",
        "--source-fp", "data/synthetic/false_positive.crdownload",
        "--source-review", "data/synthetic/review.crdownload",
    ]
if PER_CLASS_CAP:
    process_cmd.extend(["--per-class-cap", str(PER_CLASS_CAP)])

run_cmd(process_cmd, cwd=PROJECT_DIR)

In [ ]:
import json

def count_lines(path):
    with open(path, "r", encoding="utf-8") as handle:
        return sum(1 for _ in handle)

report = json.loads(Path("data/augmentation_report.json").read_text(encoding="utf-8"))
print(json.dumps(report, indent=2)[:3000])

for split_path in ["data/training_data_binary.jsonl", "data/val_data_binary.jsonl", "data/test_data_binary.jsonl"]:
    print(split_path, count_lines(split_path))

## 5. Train LoRA Adapter Models

Trains a LoRA adapter for every model with `"enabled": True` in `MODELS`. Skipped entirely when `RUN_TRAINING = False`. Enable one model first; add the others once a run succeeds and the GPU has headroom. Set `LOAD_4BIT = True` to fine-tune with QLoRA 4-bit on smaller GPUs.

In [ ]:
trained_runs = []

if not RUN_TRAINING:
    print("RUN_TRAINING is False - skipping LoRA training.")
else:
    for run in MODELS:
        if not run.get("enabled", False):
            print("Skipping disabled model:", run["ollama_name"])
            continue

        output_dir = Path(run["output_dir"])
        if output_dir.exists():
            shutil.rmtree(output_dir)

        train_cmd = [
            sys.executable,
            "scripts/lora_fine_tune.py",
            "--model", run["ollama_name"],
            "--epochs", str(EPOCHS),
            "--batch-size", str(BATCH_SIZE),
            "--learning-rate", str(LEARNING_RATE),
            "--gpu",
            "--target", "both",
            "--train", "data/training_data_binary.jsonl",
            "--val", "data/val_data_binary.jsonl",
            "--output-dir", str(output_dir),
        ]
        if LOAD_4BIT:
            train_cmd.append("--load-4bit")
        run_cmd(train_cmd, cwd=PROJECT_DIR)
        trained_runs.append(run)

print("Trained runs:", [run["ollama_name"] for run in trained_runs])

## 6. Evaluate Adapters

Evaluation loads the Hugging Face base model plus the LoRA adapter and writes JSON reports to `/kaggle/working`. Use `EVAL_LIMIT = 0` for full evaluation after a smoke run succeeds.

In [ ]:
evaluation_reports = []

if not RUN_EVALUATION:
    print("RUN_EVALUATION is False - skipping adapter evaluation.")
else:
    for run in trained_runs:
        safe_name = run["ollama_name"].replace(":", "-").replace("/", "-")
        report_path = Path(f"/kaggle/working/evaluation_{safe_name}.json")
        eval_cmd = [
            sys.executable,
            "scripts/evaluate_model_performance.py",
            "--base-model", run["hf_name"],
            "--adapter", run["output_dir"],
            "--test", "data/test_data_binary.jsonl",
            "--report", str(report_path),
        ]
        if EVAL_LIMIT:
            eval_cmd.extend(["--limit", str(EVAL_LIMIT)])
        run_cmd(eval_cmd, cwd=PROJECT_DIR)
        evaluation_reports.append(report_path)

evaluation_reports

In [ ]:
for report_path in evaluation_reports:
    metrics = json.loads(report_path.read_text(encoding="utf-8"))
    print("\n", report_path.name)
    print("n:", metrics.get("n"))
    print("binary:", metrics.get("binary"))
    print("json_validity_rate:", metrics.get("json_validity_rate"))
    print("avg_latency_ms:", metrics.get("avg_latency_ms"))

## 7. Benchmark Reasoning Strategies

Runs `scripts/benchmark_models.py` over the `MODELS` x `STRATEGIES` matrix and writes a per-record JSONL plus an aggregated summary (binary F1, hard-negative recall, JSON / schema validity, evidence grounding, latency percentiles, escalation rate).

`BENCHMARK_TEST_MODE = True` uses deterministic fake LLM responses — no Ollama needed, which is the fast way to smoke-test that every (model, strategy) cell runs. For a real benchmark, install and serve Ollama with the enabled models pulled (`ollama pull qwen2.5-coder:3b` ...), then set `BENCHMARK_TEST_MODE = False`.

In [ ]:
benchmark_matrix_path = Path("/kaggle/working/benchmark_matrix.jsonl")
benchmark_summary_path = Path("/kaggle/working/benchmark_summary.json")

if not RUN_STRATEGY_BENCHMARK:
    print("RUN_STRATEGY_BENCHMARK is False - skipping strategy benchmark.")
else:
    benchmark_models = [m["ollama_name"] for m in MODELS if m.get("enabled")]
    if not benchmark_models:
        raise ValueError("No models enabled in MODELS - enable at least one to benchmark.")
    if not STRATEGIES:
        raise ValueError("STRATEGIES is empty - select at least one reasoning strategy.")

    benchmark_cmd = [
        sys.executable,
        "scripts/benchmark_models.py",
        "--models", *benchmark_models,
        "--strategies", *STRATEGIES,
        "--test", "data/test_data_binary.jsonl",
        "--output", str(benchmark_matrix_path),
        "--summary", str(benchmark_summary_path),
    ]
    if BENCHMARK_TEST_MODE:
        benchmark_cmd.append("--test-mode")
    if BENCHMARK_LIMIT:
        benchmark_cmd.extend(["--limit", str(BENCHMARK_LIMIT)])

    print(f"Benchmarking {len(benchmark_models)} model(s) x {len(STRATEGIES)} strategy(ies)"
          f" = {len(benchmark_models) * len(STRATEGIES)} cells"
          f" ({'test-mode' if BENCHMARK_TEST_MODE else 'real Ollama'})")
    run_cmd(benchmark_cmd, cwd=PROJECT_DIR)

In [ ]:
if RUN_STRATEGY_BENCHMARK and benchmark_summary_path.exists():
    summary = json.loads(benchmark_summary_path.read_text(encoding="utf-8"))
    print("Strategy benchmark summary (model x strategy):\n")
    print(json.dumps(summary, indent=2)[:4000])
else:
    print("No benchmark summary to display (RUN_STRATEGY_BENCHMARK off or file missing).")

## 8. Package Outputs For Download

Kaggle preserves `/kaggle/working` outputs when you save a notebook version. This cell creates one zip containing trained adapters, evaluation reports, and the strategy-benchmark results.

In [ ]:
import zipfile

artifact_zip = Path("/kaggle/working/cred_hunt_v2_lora_outputs.zip")
if artifact_zip.exists():
    artifact_zip.unlink()

with zipfile.ZipFile(artifact_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for run in trained_runs:
        adapter_dir = Path(run["output_dir"])
        if adapter_dir.exists():
            for file_path in adapter_dir.rglob("*"):
                if file_path.is_file():
                    archive.write(file_path, file_path.relative_to(adapter_dir.parent))
    for report_path in evaluation_reports:
        if report_path.exists():
            archive.write(report_path, report_path.name)
    # Strategy-benchmark outputs (defined in Section 7).
    for extra in [globals().get("benchmark_matrix_path"), globals().get("benchmark_summary_path")]:
        if extra is not None and Path(extra).exists():
            archive.write(extra, Path(extra).name)

print("Created:", artifact_zip)
print("Size MB:", artifact_zip.stat().st_size / 1024 / 1024)

## 9. After Training

Download `cred_hunt_v2_lora_outputs.zip` from Kaggle outputs. The zip contains each LoRA adapter directory, the evaluation reports, and the strategy-benchmark matrix + summary. Keep generated JSONL/CSV files out of Git unless you intentionally want to version a fixed training split.